# Practice 4 · Recognize your photo

| Marker | Meaning |
|---|---|
| 🟢 EDIT | change this line |
| 🟡 READ | read it, no need to change |
| ⚪ SKIP | run it as it is |

### I. Background

A model trained by someone else can be used directly: load the weights, run one forward pass on an image of your own, and read the probabilities. ResNet18 was trained on ImageNet, which contains 1000 everyday classes, so it can name what it sees — and it also fails on images that do not look like the ones it was trained on. In this practice you recognize your own photo, and then look for an image the model gets wrong.

### II. Task Requirements

1. Set `SID` to the last two digits of your student ID, and write the file name of your photo in `IMG_PATH` (a phone photo, a screenshot or a meme all work; if no image is found, a sample one is created).
2. Resize the photo to 224 x 224 and prepare the tensor that is sent to the model.
3. Load the pretrained model, run one forward pass, turn the 1000 scores into probabilities and print the top-3.
4. Save your photo next to a bar chart of its top-3 as `top3_<SID>.png`.
5. Find one image whose top-1 probability is below 0.5 and post it to the leaderboard (Question 4).

Steps: open HaiMa, open this notebook, fill in the missing code and run it.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| `IMG`, `img` | your photo | an RGB image that can be read |
| `x` | the tensor sent to the model | shape `(1, 3, 224, 224)` |
| `probs` | the 1000 class probabilities | shape `(1, 1000)`, sum about 1 |
| `top3` | the three largest probabilities | sorted from the largest down |
| `top3_<SID>.png` | photo + bar chart | one file |
| leaderboard | one failure image and its top-1 probability | top-1 below 0.5 |


## Question 1 · Upload your photo and prepare it

Write the file name of your photo below (a phone photo, a screenshot or a meme all work).
The next cell picks a usable image; if it finds none, it draws a sample one, so the notebook always runs.


In [10]:
# 🟢 EDIT: set SID to the last two digits of your student ID
# YOUR CODE HERE
SID = 23

# 🟢 EDIT: upload one photo next to this notebook and write its file name here
# IMG_PATH = "my_photo.jpg"
# YOUR CODE HERE
IMG_PATH = "my_photo.jpg"

import os, glob, torch
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True   # tolerate slightly-truncated uploads

from torchvision import transforms
import torchvision.models as models

found = [p for p in [IMG_PATH] if os.path.exists(p)]
if not found:
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"):
        found += sorted(glob.glob(ext))
found = [p for p in found if os.path.basename(p) != "sample_photo.png"]

# try each candidate; keep the first one that actually loads
img, IMG = None, None
for p in found:
    try:
        img = Image.open(p).convert("RGB")
        IMG = p
        break
    except Exception as e:
        print("skipped %s: %s (%s)" % (p, type(e).__name__, e))

if IMG is None:
    from PIL import ImageDraw
    W, H = 640, 480
    _im = Image.new("RGB", (W, H))
    _d = ImageDraw.Draw(_im)
    for _y in range(H):
        _d.line([(0, _y), (W, _y)], fill=(120, 165, 225 - int(110 * _y / H)))
    _d.rectangle([0, int(H * 0.72), W, H], fill=(86, 140, 78))
    _d.rectangle([200, 210, 520, 360], fill=(240, 200, 40), outline=(110, 80, 0), width=4)
    _d.ellipse([240, 330, 320, 400], fill=(45, 45, 45))
    _d.ellipse([400, 330, 480, 400], fill=(45, 45, 45))
    IMG = "sample_photo.png"
    _im.save(IMG)
    img = Image.open(IMG).convert("RGB")
    print("no usable image found -> a sample image was created:", IMG)
else:
    print("using:", IMG)

print("original size:", img.size, " mode:", img.mode)

using: my_photo.jpg
original size: (1440, 1577)  mode: RGB


## Question 2 · Resize to 224 × 224 and run one forward pass

The preprocessing and the pretrained model are given; fill in the two missing lines: the tensor sent to the model, and the softmax that turns the scores into probabilities.


In [11]:
prep = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 🟢 EDIT: prepare the tensor sent to the model (one image in a batch). You may need prep(img) to resize/crop/normalize, then .unsqueeze(0) to add the batch dimension — expect (1, 3, 224, 224)
# YOUR CODE HERE
x = prep(img).unsqueeze(0)
print("tensor sent to the model:", tuple(x.shape), " (expected (1, 3, 224, 224))")

weights = models.ResNet18_Weights.DEFAULT
classes = weights.meta["categories"]   # the 1000 class names, in the model's order
try:
    model = models.resnet18(weights=weights).eval()
    print("model ready: ResNet18 with pretrained weights")
except Exception as e:
    print("pretrained weights unavailable:", type(e).__name__, str(e)[:120])
    model = models.resnet18(weights=None).eval()
    print("continuing with random weights: the probabilities below are meaningless")

with torch.no_grad():
    logits = model(x)

# 🟢 EDIT: turn the 1000 scores into probabilities (they should sum to 1). You may need torch.softmax(logits, dim=1) — dim=1 normalizes across the 1000 classes
# YOUR CODE HERE
probs = torch.softmax(logits, dim=1)


top5 = torch.topk(probs, 5)
print("sum of probabilities =", float(probs.sum()), " (should be about 1)")
for p, i in zip(top5.values[0], top5.indices[0]):
    print("  %s (class #%d)  probability %.3f" % (classes[int(i)], int(i), float(p)))


tensor sent to the model: (1, 3, 224, 224)  (expected (1, 3, 224, 224))
model ready: ResNet18 with pretrained weights
sum of probabilities = 0.9999998807907104  (should be about 1)
  crate (class #519)  probability 0.349
  washer (class #897)  probability 0.079
  dishwasher (class #534)  probability 0.057
  photocopier (class #713)  probability 0.046
  washbasin (class #896)  probability 0.045


## Question 3 · Save your photo with its top-5

Draw the photo next to a bar chart of its top-3 predictions and save the figure as `top5_<SID>.png`.


In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.4))
a1.imshow(img); a1.axis("off"); a1.set_title("My photo (SID %d)" % SID)
k = [int(i) for i in top5.indices[0]]
v = [float(t) for t in top5.values[0]]
a2.barh([classes[i] for i in k][::-1], v[::-1], color="#2E6DB4")
a2.tick_params(axis="y", labelsize=9)
a2.set_xlabel("probability"); a2.set_xlim(0, 1)
a2.set_title("Top-5 classes — SID %d" % SID)

# 🟢 EDIT: save the figure as top5_<SID>.png
# YOUR CODE HERE
plt.savefig("top5_%d.png" % SID, dpi=110)
plt.tight_layout()
plt.show()


C:\Users\Administrator\AppData\Local\Temp\ipykernel_5752\1234086768.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
